# modulos/upsert — `upsert_delta_table` (gravação Delta multi-edição)

Módulo carregado pelo hub `environment/env_functions`. Função única de gravação do pipeline: todo notebook usa esta função para **criar ou atualizar** tabelas Delta, sempre particionadas por `ano_edicao` — é o que permite edições diferentes (2019, 2024...) conviverem na mesma tabela sem se apagarem.

**Assinatura:** `upsert_delta_table(df_source, table_name, merge_keys, ano_edicao, description)`

| Parâmetro | O que é |
| --- | --- |
| `df_source` | DataFrame Spark a ser gravado |
| `table_name` | destino completo (`catalog.schema.tabela`) |
| `merge_keys` | chaves do MERGE, ex. `["cod_munic", "ano_edicao"]` |
| `ano_edicao` | ano da edição corrente (vira coluna e partição) |
| `description` | texto gravado no histórico Delta (`DESCRIBE HISTORY` → `userMetadata`) |

**A função executa 6 passos, sempre nesta ordem — cada passo é uma função auxiliar com o mesmo nome, logo abaixo da principal:**

1. **Completar a origem** (`_adicionar_ano_edicao_na_origem`) — se o DataFrame ainda não traz `ano_edicao`, adiciona (com o ano da chamada) e o posiciona como 2ª coluna, logo após a chave da entidade.
2. **Tabela nova?** (`_criar_tabela_particionada`) — se o destino não existe no catálogo, cria já particionada por `ano_edicao` e **encerra aqui** (não há nada a atualizar).
3. **Tabela existente no formato multi-edition?** — uma de duas coisas: **legado sem `ano_edicao`** (`_migrar_para_particionada` — regrava a tabela particionada; a partição nasce de `reference_year`, coluna de cargas antigas, ou do parâmetro `ano_edicao`) ou **já particionada** (`_reordenar_para_padrao` — apenas garante a ordem visual `chave da entidade, ano_edicao, resto`, sem reescrever nada se já estiver assim).
4. **Evolução de schema** (`_adicionar_colunas_novas`) — colunas que existem na origem e não no destino entram via `ALTER TABLE ADD COLUMN` (ex.: edição nova traz pergunta nova, como `mtic018`).
5. **Alinhamento reverso** (`_preencher_colunas_do_destino_com_null`) — colunas que existem só no destino (herdadas de outras edições) recebem `null` na origem: o MERGE exige que toda coluna do destino exista na origem.
6. **MERGE** (`_executar_merge`) — atualiza as linhas que casam pelas `merge_keys` e insere as demais.

**Por que as `merge_keys` sempre incluem `ano_edicao`** (ex.: `["cod_munic", "ano_edicao"]`): o MERGE compara linha a linha por essas chaves, então a edição de 2024 só atualiza linhas de 2024 — nunca sobrescreve as de 2019.

**Auditoria:** toda escrita (criação, migração e MERGE) registra o texto da operação no log Delta — visível em `DESCRIBE HISTORY <tabela>`, coluna `userMetadata` (em compute serverless o `_set_user_metadata` tenta, avisa uma vez e segue).

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F


def upsert_delta_table(df_source, table_name, merge_keys, ano_edicao, description):
    """Upsert de um Spark DataFrame em uma tabela Delta particionada por ano_edicao.

    Executa 6 passos, sempre nesta ordem (cada um é uma função auxiliar logo abaixo,
    com o mesmo nome — o markdown acima detalha o que cada passo faz):
      1. _adicionar_ano_edicao_na_origem   — completa a origem com a coluna de partição
      2. _criar_tabela_particionada        — tabela nova: cria e encerra
      3. _migrar_para_particionada OU
         _reordenar_para_padrao            — tabela existente no formato multi-edition
      4. _adicionar_colunas_novas          — evolução de schema (ALTER TABLE ADD)
      5. _preencher_colunas_do_destino_com_null — alinhamento reverso (null na origem)
      6. _executar_merge                    — MERGE: atualiza o que casou, insere o resto
                                            (userMetadata via conf de sessão — seção 6 do
                                             desenho do pipeline_manifest)

    Parameters
    ----------
    df_source    : Spark DataFrame a ser gravado (upsert)
    table_name   : nome totalmente qualificado da tabela de destino
    merge_keys   : lista de colunas para a condição do MERGE
    ano_edicao   : ano da edição da pesquisa (int)
    description  : string gravada no userMetadata da operação Delta
    """
    # ---- PASSO 1: a origem precisa trazer a coluna de partição 'ano_edicao' ----
    df_source, source_cols = _adicionar_ano_edicao_na_origem(df_source, ano_edicao)

    # ---- PASSO 2: tabela nova? cria particionada e encerra por aqui ----
    if not spark.catalog.tableExists(table_name):
        _criar_tabela_particionada(df_source, table_name, ano_edicao, description)
        return
    deltaTable = DeltaTable.forName(spark, table_name)

    # ---- PASSO 3: tabela existente -> garante o formato multi-edition ----
    if "ano_edicao" not in deltaTable.toDF().columns:
        # 3a. legado (criada antes da partição por edição): deriva a partição e regrava
        deltaTable, target_cols = _migrar_para_particionada(deltaTable, table_name, ano_edicao)
    else:
        # 3b. já particionada: só garante a ordem de colunas do projeto
        deltaTable, target_cols = _reordenar_para_padrao(deltaTable, table_name, merge_keys)

    # ---- PASSO 4: evolução de schema (colunas novas na origem) ----
    if _adicionar_colunas_novas(table_name, source_cols, target_cols):
        deltaTable = DeltaTable.forName(spark, table_name)  # recarrega após o ALTER

    # ---- PASSO 5: alinhamento reverso (colunas só no destino viram null na origem) ----
    df_source = _preencher_colunas_do_destino_com_null(df_source, deltaTable, source_cols, table_name)

    # ---- PASSO 6: MERGE (atualizar o que casou, inserir o que é novo) ----
    _executar_merge(deltaTable, df_source, merge_keys, table_name, ano_edicao, description)


# ---- PASSO 1 — completar a origem com a coluna de partição -----------------
def _adicionar_ano_edicao_na_origem(df_source, ano_edicao):
    """Se 'ano_edicao' não existir na origem, adiciona e a posiciona como 2ª coluna
    (logo após a chave da entidade — ordem visual padrão do projeto).

    Retorna (df, colunas): 'colunas' reflete a ordem final da origem e é
    reaproveitada nos passos 4 e 5.
    """
    if "ano_edicao" in df_source.columns:
        return df_source, list(df_source.columns)

    primeira, restantes = df_source.columns[0], df_source.columns[1:]
    df_source = (
        df_source
        .withColumns({"ano_edicao": F.lit(ano_edicao)})
        .select(primeira, "ano_edicao", *restantes)
    )
    return df_source, [primeira, "ano_edicao", *restantes]


# ---- PASSO 2 — criação da tabela (primeira gravação) -----------------------
def _criar_tabela_particionada(df_source, table_name, ano_edicao, description):
    """Cria a tabela já no formato multi-edition: particionada por 'ano_edicao',
    com 'description' registrada no histórico Delta (userMetadata)."""
    (df_source.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("ano_edicao")
        .option("userMetadata", description)
        .saveAsTable(table_name))
    print(f"Criada: {table_name} - ano {ano_edicao}")


# ---- PASSO 3 — tabela existente no formato multi-edition --------------------
def _migrar_para_particionada(deltaTable, table_name, ano_edicao):
    """Tabela legado sem 'ano_edicao': deriva a partição e regrava a tabela.

    - Se a tabela já tem 'reference_year' (cargas antigas), a partição nasce dele;
    - senão usa o 'ano_edicao' passado (ex.: tabela de dicionário, sem ano por linha).
    Regrava com 'ano_edicao' como 2ª coluna — resultado idêntico a uma tabela nova.

    Retorna o DeltaTable recarregado e as colunas resultantes.
    """
    print(f"Migrando {table_name}: adicionando a coluna de partição ano_edicao...")
    # Materializa o DataFrame uma vez para evitar RPCs de Analyze repetidos no Spark Connect
    target_df = deltaTable.toDF()
    target_cols = target_df.columns

    if "reference_year" in target_cols:
        valor_particao = F.col("reference_year").cast("int")   # partição nasce da coluna existente
    else:
        valor_particao = F.lit(ano_edicao)                     # sem reference_year -> usa o parâmetro

    primeira, restantes = target_cols[0], target_cols[1:]
    existing_df = target_df.withColumns({"ano_edicao": valor_particao}).select(
        primeira, "ano_edicao", *restantes
    )

    (existing_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("ano_edicao")
        .option("userMetadata", "Migracao: coluna de particao ano_edicao adicionada")
        .saveAsTable(table_name))

    return DeltaTable.forName(spark, table_name), [primeira, "ano_edicao", *restantes]


def _reordenar_para_padrao(deltaTable, table_name, merge_keys):
    """Garante a ordem de colunas do projeto: chave da entidade em 1º, 'ano_edicao' em 2º.

    Se já estiver nessa ordem, não faz nada (evita ALTER desnecessário).
    """
    target_cols = deltaTable.toDF().columns
    entity_key = next(k for k in merge_keys if k != "ano_edicao")

    ja_ordenado = (
        len(target_cols) >= 2
        and target_cols[0] == entity_key
        and target_cols[1] == "ano_edicao"
    )
    if ja_ordenado:
        return deltaTable, target_cols

    # ALTER FIRST duas vezes, nessa ordem: o resultado é [entity_key, ano_edicao, resto]
    spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN ano_edicao FIRST")
    spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN {entity_key} FIRST")
    target_cols = [entity_key, "ano_edicao"] + [
        c for c in target_cols if c not in (entity_key, "ano_edicao")
    ]
    return DeltaTable.forName(spark, table_name), target_cols


# ---- PASSO 4 — evolução de schema (colunas novas na origem) -----------------
def _adicionar_colunas_novas(table_name, source_cols, target_cols):
    """Colunas que existem na origem mas não no destino: ALTER TABLE ADD COLUMN.

    Retorna True se alguma coluna foi adicionada (sinal para recarregar o DeltaTable).
    """
    faltantes = [c for c in source_cols if c not in target_cols]
    if not faltantes:
        return False
    for col in faltantes:
        spark.sql(f"ALTER TABLE {table_name} ADD COLUMN {col} STRING")
    print(f"Colunas adicionadas a {table_name}: {faltantes}")
    return True


# ---- PASSO 5 — alinhamento reverso (colunas só no destino) -----------------
def _preencher_colunas_do_destino_com_null(df_source, deltaTable, source_cols, table_name):
    """Colunas que existem no destino mas não na origem: recebem null na origem.

    Necessário porque whenMatchedUpdateAll/whenNotMatchedInsertAll exigem que TODAS
    as colunas do destino sejam resolvíveis na origem — em upserts entre edições da
    pesquisa, o destino carrega colunas de outras edições (ex.: mtic018 existe em
    2024, mas não em 2019).
    """
    target_schema = {f.name: f.dataType for f in deltaTable.toDF().schema.fields}
    so_no_destino = [c for c in target_schema if c not in source_cols]
    if not so_no_destino:
        return df_source

    df_source = df_source.withColumns({
        c: F.lit(None).cast(target_schema[c].simpleString()) for c in so_no_destino
    })
    print(f"Colunas do destino ausentes na origem, preenchidas com null em {table_name}:\n {so_no_destino}\n")
    return df_source


# ---- PASSO 6 — MERGE (atualizar o que casou, inserir o que é novo) ----------
_USER_METADATA_OK = None  # None = ainda não testado; True/False após a 1a tentativa


def _set_user_metadata(description):
    """userMetadata do MERGE: só chega ao histórico via conf de sessão — MAS em
    compute serverless essa conf é CONFIG_NOT_AVAILABLE (SQLSTATE 42K0I,
    verificado em 2026-11-13). Tenta setar; quando não der, avisa UMA vez e
    segue (a trilha do commit continua com operation/timestamp do histórico)."""
    global _USER_METADATA_OK
    if _USER_METADATA_OK is False:
        return False
    try:
        spark.conf.set("spark.databricks.delta.commitInfo.userMetadata", description)
        _USER_METADATA_OK = True
        return True
    except Exception:
        _USER_METADATA_OK = False
        print("AVISO: userMetadata indisponível neste compute (serverless) — os MERGEs "
              "seguem sem a descrição no DESCRIBE HISTORY.")
        return False


def _executar_merge(deltaTable, df_source, merge_keys, table_name, ano_edicao, description=""):
    """MERGE pelas merge_keys: 'target.<k> = source.<k> AND ...'.

    Ex.: ["cod_munic", "ano_edicao"] -> a edição de 2024 atualiza apenas as
    linhas de 2024, nunca as de 2019 — é isso que mantém as edições independentes.

    userMetadata: o MERGE pela API do Delta ignora a option do write — a
    descrição só chega ao histórico (DESCRIBE HISTORY) via conf de sessão
    (docs/arquitetura/pipeline_manifest.md, seção 6). Em compute SERVERLESS
    essa conf não existe (CONFIG_NOT_AVAILABLE, verificado em 2026-11-13):
    o _set_user_metadata tenta, avisa uma vez e segue sem interromper.
    """
    if description:
        _set_user_metadata(description)
    merge_condition = " AND ".join(f"target.{k} = source.{k}" for k in merge_keys)

    (deltaTable.alias("target")
        .merge(df_source.alias("source"), merge_condition)
        .whenMatchedUpdateAll()      # linha que casou -> atualiza
        .whenNotMatchedInsertAll()   # linha que não casou -> insere
        .execute())
    print(f"MERGE concluído: {table_name} - ano {ano_edicao}")
